# Chapter 5 · Memory Systems for Agents
**Mastering Agentic AI** — Manning Publications



## What this notebook covers
- The four types of agent memory and when to use each
- SemanticMemory: structured user facts persisted to disk
- EpisodicMemory: session summaries with keyword-based retrieval
- InContextMemory: sliding-window conversation management
- RAG pattern: retrieve relevant episodes before each response
- Memory-enabled Diet Coach using LangChain + Mem0

---

## 5.0 · Setup

In [ ]:
# !pip install openai python-dotenv langchain-openai mem0ai
import os, json, datetime, hashlib, statistics
from pathlib import Path
from typing import Any, List, Dict
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from mem0 import MemoryClient
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY — see appendix_a_api_keys.md"
client = OpenAI()
MODEL = "gpt-4o-mini"
print("Ready")

## 5.1 · The Four Types of Agent Memory

| Type | What it stores | Lifetime | Production store |
|---|---|---|---|
| **In-context** | Current conversation turns | Session only | Context window (list of dicts) |
| **Episodic** | Summaries of past sessions | Persistent | Vector DB (FAISS, Pinecone) |
| **Semantic** | User facts and preferences | Persistent | Key-value store (Redis) |
| **Procedural** | Skills and workflows | Persistent | SKILL.md files (Chapter 3) |

The key insight: each type solves a different problem. In-context memory is fastest but ephemeral. Episodic and semantic memory persist but require retrieval. Procedural memory (Skills) is injected at agent startup and rarely changes.



## 5.2 · SemanticMemory — Storing User Facts

In [ ]:
class SemanticMemory:
    """
    Stores structured facts about a user that persist across sessions.

    Production upgrade path:
      JSON file (this chapter) → Redis → DynamoDB
    The interface stays identical — only the backend changes.
    """
    def __init__(self, user_id: str, store_path: Path = Path(".memory")):
        self.user_id = user_id
        self.path = store_path / f"{user_id}_semantic.json"
        self.path.parent.mkdir(exist_ok=True)
        self._data: dict = json.loads(self.path.read_text()) if self.path.exists() else {}

    def set(self, key: str, value: Any) -> None:
        self._data[key] = value
        self.path.write_text(json.dumps(self._data, indent=2))

    def get(self, key: str, default: Any = None) -> Any:
        return self._data.get(key, default)

    def all(self) -> dict:
        return dict(self._data)

    def as_context_string(self) -> str:
        """Formats memory as a human-readable context block for the system prompt."""
        if not self._data:
            return "No user profile stored yet."
        lines = [f"  {k}: {v}" for k, v in self._data.items()]
        return "User profile (from semantic memory):\n" + "\n".join(lines)

# Test SemanticMemory
mem = SemanticMemory("alex_demo")
mem.set("name", "Alex")
mem.set("weight_kg", 78)
mem.set("goal", "lose 5kg")
mem.set("restriction", "lactose intolerant")
mem.set("activity", "moderate — 3 gym sessions per week")
print(mem.as_context_string())

## 5.3 · EpisodicMemory — Session Summaries

In [ ]:
class EpisodicMemory:
    """
    Stores summaries of past coaching sessions.

    Each episode: {date, summary, key_insights, goal_set}

    Production upgrade: embed summaries with sentence-transformers,
    index with FAISS. The retrieve() method becomes a semantic search
    instead of a keyword search.
    """
    def __init__(self, user_id: str, store_path: Path = Path(".memory")):
        self.user_id = user_id
        self.path = store_path / f"{user_id}_episodes.json"
        self.path.parent.mkdir(exist_ok=True)
        self._episodes: list[dict] = json.loads(self.path.read_text()) if self.path.exists() else []

    def add_episode(self, summary: str, key_insights: list[str], goal_set: str | None = None) -> None:
        episode = {
            "date":         datetime.date.today().isoformat(),
            "summary":      summary,
            "key_insights": key_insights,
            "goal_set":     goal_set,
            "id":           hashlib.md5(f"{self.user_id}{datetime.datetime.now()}".encode()).hexdigest()[:8],
        }
        self._episodes.append(episode)
        self.path.write_text(json.dumps(self._episodes, indent=2))
        return episode

    def retrieve(self, query: str, n: int = 3) -> list[dict]:
        """Keyword-based retrieval — replace with vector search in production."""
        query_words = set(query.lower().split())
        scored = []
        for ep in self._episodes:
            text = f"{ep['summary']} {' '.join(ep.get('key_insights', []))}".lower()
            overlap = len(query_words & set(text.split()))
            scored.append((overlap, ep))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [ep for _, ep in scored[:n]]

    def get_recent(self, n: int = 3) -> list[dict]:
        return self._episodes[-n:]

    def as_context_string(self, query: str | None = None, n: int = 3) -> str:
        episodes = self.retrieve(query, n) if query else self.get_recent(n)
        if not episodes:
            return "No past sessions recorded."
        lines = []
        for ep in episodes:
            lines.append(f"  [{ep['date']}] {ep['summary']}")
            if ep.get("goal_set"):
                lines.append(f"    Goal: {ep['goal_set']}")
        return "Relevant past sessions:\n" + "\n".join(lines)

# Simulate past sessions
ep_mem = EpisodicMemory("alex_demo")
ep_mem.add_episode(
    summary="Alex set a 5kg weight loss goal. Identified protein deficit (only 45g/day vs 62g target).",
    key_insights=["low protein intake", "skips breakfast", "high fibre from vegetables"],
    goal_set="Eat 30g of protein at breakfast every day this week."
)
ep_mem.add_episode(
    summary="Alex reported completing the breakfast goal 5/7 days. Energy levels improved.",
    key_insights=["improved adherence", "energy improvement", "still skips breakfast 2 days"],
    goal_set="Add a protein shake on the two days when breakfast is skipped."
)
print(ep_mem.as_context_string(query="protein breakfast"))

## 5.4 · InContextMemory — Sliding Window Management

In [ ]:
class InContextMemory:
    """
    Manages the current conversation with sliding window pruning.
    Prevents context window overflow without losing recent turns.
    """
    def __init__(self, max_turns: int = 20):
        self.max_turns = max_turns
        self._history: list[dict] = []

    def add(self, role: str, content: Any) -> None:
        self._history.append({"role": role, "content": content})

    def get_window(self) -> list[dict]:
        # 2 messages per turn (user + assistant)
        return self._history[-(self.max_turns * 2):]

    def clear(self) -> None:
        self._history = []

    def __len__(self) -> int:
        return len(self._history)

ctx = InContextMemory(max_turns=3)
ctx.add("user", "Hi, I want to lose weight")
ctx.add("assistant", "Great! Let me help you with that.")
ctx.add("user", "What should I eat for breakfast?")
ctx.add("assistant", "I recommend eggs and oats.")
ctx.add("user", "Thanks! What about lunch?")
ctx.add("assistant", "For lunch, try chicken and vegetables.")
# Only keeps last 3 turns (6 messages)
print(f"History: {len(ctx)} messages | Window: {len(ctx.get_window())} messages")

## 5.2a · Compaction — When the Window Fills

Sliding the window drops old turns but **loses their information**. Compaction is the alternative: compress history before it overflows, preserving meaning while reducing token count.

Three patterns cover most cases:

| Pattern | How it works | When to use it |
|---------|-------------|----------------|
| **Result-only** | Keep outcomes, strip working detail | Coding/research agents with verbose tool outputs |
| **LLM summarisation** | Compress history with a separate model call | Complex multi-topic sessions |
| **Ephemeral context** | Append once, never store — see Section 5.6a | Real-time signals that change faster than the conversation |

The Diet Coach already uses compaction **implicitly**: episodic summaries replace raw transcripts, and `build_context_window` includes only the last 3 meals. This section names and implements the pattern explicitly.

In [ ]:
def compact_history_result_only(
    history: list[dict],
    result_roles: tuple[str, ...] = ('assistant',),
) -> list[dict]:
    """
    Section 5.2a: Result-only compaction.
    Keep the outcome of each exchange; discard intermediate working detail.
    In a coding agent: strip compiler logs, keep 'compile successful'.
    In a research agent: strip training curves, keep validation loss.
    """
    return [msg for msg in history if msg.get('role') in result_roles]


def compact_history_llm_summary(
    history: list[dict],
    client,
    model: str = 'gpt-4o-mini',
) -> list[dict]:
    """
    Section 5.2a: LLM-summarisation compaction.
    Compress the full history into a single system message when
    result-only compaction would lose important reasoning steps.
    """
    if not history:
        return []
    transcript = '\n'.join(
        f"{m['role'].upper()}: {str(m['content'])[:300]}" for m in history
    )
    response = client.chat.completions.create(
        model=model, max_tokens=200,
        messages=[{'role': 'user', 'content':
            'Summarise this conversation in 2-3 sentences. '
            'Retain only facts needed to continue the coaching session:\n\n'
            + transcript}],
    )
    summary = response.choices[0].message.content.strip()
    return [{'role': 'system', 'content': f'[Compacted history] {summary}'}]


# Demo — result-only compaction
sample_history = [
    {'role': 'user',      'content': 'What should I eat for breakfast?'},
    {'role': 'assistant', 'content': 'Try eggs and smoked salmon — high protein.'},
    {'role': 'user',      'content': 'What about lunch?'},
    {'role': 'assistant', 'content': 'A chicken salad would keep you on track.'},
]
compacted = compact_history_result_only(sample_history)
print(f'Original: {len(sample_history)} turns → Compacted: {len(compacted)} turns')
for msg in compacted:
    print(f"  [{msg['role']}] {msg['content']}")


## 5.4a · Writable Skills — Procedural Memory the Agent Can Update

Procedural memory (`SKILL.md`) is the **one memory type the agent can rewrite**.

- Semantic, episodic, and in-context memory are written by the surrounding system — they record *what happened*.
- Procedural memory records *how the agent should think* — and the agent can update that from experience.

A skill is just a text file. If the agent observes that a different approach consistently produces better outcomes, it writes a revised instruction into `SKILL.md`. The change propagates to every future session with **no fine-tuning, no redeployment** — just a file write.

This is the foundation for **self-evolving agents** (covered in Chapter 9).

In [ ]:
def read_skill(skill_path: Path) -> str:
    """Section 5.4a: Load the current procedural skill from disk."""
    return skill_path.read_text(encoding='utf-8') if skill_path.exists() else ''


def update_skill(skill_path: Path, new_content: str, backup: bool = True) -> None:
    """
    Section 5.4a: Overwrite the skill file with updated content.
    If backup=True, saves previous version to SKILL.md.bak.
    """
    skill_path.parent.mkdir(parents=True, exist_ok=True)
    if backup and skill_path.exists():
        skill_path.with_suffix('.md.bak').write_text(
            skill_path.read_text(encoding='utf-8'), encoding='utf-8'
        )
    skill_path.write_text(new_content, encoding='utf-8')
    print(f'[skill] Updated: {skill_path}')


def append_skill_note(skill_path: Path, note: str) -> None:
    """
    Section 5.4a: Append a learned heuristic to the existing skill
    without replacing the whole protocol.
    """
    current = read_skill(skill_path)
    updated = current.rstrip() + f'\n\n# Agent observation\n{note.strip()}\n'
    update_skill(skill_path, updated, backup=True)
    print(f'[skill] Note appended to {skill_path.name}')


# Demo — write and update a skill
demo_skill_path = Path('.memory/demo_SKILL.md')
demo_skill_path.parent.mkdir(exist_ok=True)
demo_skill_path.write_text('# Demo Skill\nStep 1: Establish baseline.\nStep 2: Identify deficits.\n')

print('Before update:')
print(read_skill(demo_skill_path))

append_skill_note(
    demo_skill_path,
    'Users respond better when activity level is asked before discussing dietary deficits.'
)

print('After update:')
print(read_skill(demo_skill_path))


## 5.5 · Memory for our Diet Coach (LangChain + Mem0)

Memory is not a feature of the LLM. It is an external system design choice. The LLM itself remains stateless — every illusion of memory is created by retrieving past information and injecting it into the model's context at generation time.

The loop:
1. **User Input** → Retrieve relevant memories (long-term)
2. **Inject** memories into the prompt
3. **LLM generates** response
4. **Persist** new information back to memory

In [ ]:
# Step 1: Initialise the model and memory client
llm = ChatOpenAI(model="gpt-4.1-nano")
memory_client = MemoryClient()

# Step 2: Prompt with a reserved slot for memory-derived context
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content="""
You are a helpful AI Diet Coach.
Use the provided context to remember user preferences, dietary restrictions,
and long-term goals. Base your recommendations on this information.
"""),
    MessagesPlaceholder(variable_name="context"),
    HumanMessage(content="{input}")
])


def retrieve_context(query: str, user_id: str) -> List[Dict]:
    """Step 3: Retrieve relevant memories for the current query."""
    memories = memory_client.search(query, user_id=user_id)
    if not memories.get("results"):
        return [{"role": "user", "content": query}]
    serialized = " ".join(memory["memory"] for memory in memories["results"])
    return [
        {"role": "system", "content": f"Relevant user information: {serialized}"},
        {"role": "user", "content": query}
    ]


def generate_response(user_input: str, context: List[Dict]) -> str:
    """Step 4: Pass assembled context to the LLM."""
    chain = prompt | llm
    response = chain.invoke({
        "context": context,
        "input": user_input
    })
    return response.content


def save_interaction(user_id: str, user_input: str, assistant_response: str):
    """Step 5: Persist the interaction back into Mem0."""
    interaction = [
        {"role": "user", "content": user_input},
        {"role": "assistant", "content": assistant_response}
    ]
    memory_client.add(interaction, user_id=user_id)


def chat_turn(user_input: str, user_id: str) -> str:
    """Step 6: A single conversation turn — retrieve, generate, persist."""
    context = retrieve_context(user_input, user_id)
    response = generate_response(user_input, context)
    save_interaction(user_id, user_input, response)
    return response

# Demo
USER_ID = "alex_demo"

print("Week 1:")
print(chat_turn("I'm on a cut. High protein, low carbs.", USER_ID))
print("\n---\n")

print("Week 3:")
print(chat_turn("What should I have for dinner?", USER_ID))

## 5.6 · Context Engineering

Memory stores answer **what** the agent knows. Context engineering answers a different question: **what does the model see right now**, in what order, and at what level of detail?

Every token in the context window is a decision. The `build_context_window` function below assembles that window deliberately across three layers:

| Layer | Content | Why this order |
|-------|---------|----------------|
| 0 | Skill protocol | Injected via system prompt — highest attention weight |
| 1 | User profile | Stable background facts; changes rarely |
| 2 | Recent meals | Dynamic, recency-weighted; last 3 only |
| 3 | Current message | Always last — model weights recent tokens most heavily |

The **fake assistant acknowledgement turns** (`'Understood — I have your profile loaded.'`) are a deliberate priming technique. They condition the model to treat injected context as already-acknowledged information rather than summarising it back to the user.

In [ ]:
# Load the Skill protocol once at module level.
# build_context_window uses it as a default so callers don't need to
# pass it explicitly — consistent with how MemoryEnabledDietCoach loads it.
_skill_path = Path("../chapter_03_prompting/SKILL.md")
NUTRITION_ASSESSMENT_SKILL: str = (
    _skill_path.read_text() if _skill_path.exists() else ""
)


def build_context_window(
    user_message: str,
    meal_history: list[dict] | None = None,
    user_profile: dict | None = None,
    skill_text: str = NUTRITION_ASSESSMENT_SKILL,
) -> list[dict]:
    """
    Section 5.6: Assemble the message list the model will see.

    Treats every token as a deliberate decision rather than passing
    everything into the context and hoping the model pays attention
    to the right things.

    Args:
        user_message:  The user's current input — always the final message.
        meal_history:  Full meal log; only the last 3 entries are included.
        user_profile:  Dict of user facts (weight, goal, restrictions).
        skill_text:    The Nutrition Assessment Skill from SKILL.md.

    Returns:
        Message list in OpenAI API format, ordered by relevance.
    """
    messages: list[dict] = []

    # Layer 1 — User profile (stable background context)
    # Injected first; the synthetic assistant reply primes the model
    # to treat this as already-acknowledged rather than repeating it.
    if user_profile:
        profile_note = f"[Context] User profile:\n{json.dumps(user_profile, indent=2)}"
        messages.append({"role": "user",      "content": profile_note})
        messages.append({"role": "assistant", "content": "Understood — I have your profile loaded."})

    # Layer 2 — Recent meals (dynamic, recency-weighted)
    # Limit to the last 3: older entries are noise. The 'lost in the
    # middle' problem means information buried deep in long contexts
    # receives less model attention.
    if meal_history:
        recent = meal_history[-3:]
        history_note = (
            "Recent meals logged:\n"
            + "\n".join(
                f"  - {m.get('food', '?')} ({m.get('meal_type', '?')})"
                for m in recent
            )
        )
        messages.append({"role": "user",      "content": history_note})
        messages.append({"role": "assistant", "content": "Got it — I'll factor in your recent meals."})

    # Layer 3 — Current message (always last, always highest weight)
    messages.append({"role": "user", "content": user_message})
    return messages


print("build_context_window defined.")

### Demonstration — what the model actually sees

Run the cell below to inspect the assembled context window. Notice:
- The apple snack is **excluded** — it falls outside the last-3 window
- The profile and meal history arrive as **fake dialogue turns**, not raw JSON injected into the system prompt
- The user's actual question is always the **final message**

In [ ]:
user_profile = {
    "name":           "Jordan",
    "age":            32,
    "weight_kg":      78,
    "goal":           "Lose 5 kg over 3 months",
    "restrictions":   "Lactose intolerant",
    "activity_level": "Runs 3x per week",
}

meal_history = [
    {"food": "oats with banana",       "meal_type": "breakfast"},
    {"food": "chicken salad",          "meal_type": "lunch"},
    {"food": "salmon with brown rice", "meal_type": "dinner"},
    {"food": "apple",                  "meal_type": "snack"},   # dropped — outside last 3
    {"food": "eggs and smoked salmon", "meal_type": "breakfast"},
]

messages = build_context_window(
    user_message="How am I doing on protein today?",
    meal_history=meal_history,
    user_profile=user_profile,
)

print(f"Context window: {len(messages)} messages\n")
for i, msg in enumerate(messages):
    preview = msg["content"][:80] + "..." if len(msg["content"]) > 80 else msg["content"]
    print(f"  [{i}] {msg['role']:10s} | {preview}")

print("\nLayer breakdown:")
print("  Messages 0–1 : Layer 1 — user profile (stable background)")
print("  Messages 2–3 : Layer 2 — last 3 meals (chicken salad, salmon, eggs)")
print("  Message  4   : Layer 3 — current question (highest attention weight)")
print("\nNote: 'apple' excluded — falls outside the last-3 window.")

## 5.6a · Ephemeral Context — Use It Once, Discard It

`build_context_window` already uses ephemeral context implicitly — meal history and the user profile are assembled fresh every call and never written into the in-context window. This section names and generalises the pattern.

**Ephemeral context** is information that is:
- Critical for the **current** generation turn
- Not useful in subsequent turns (stale, changes rapidly, or truly one-off)

| Layer | Type | Stored in `in_context`? |
|-------|------|------------------------|
| User profile | Ephemeral (re-fetched from SemanticMemory) | ❌ Never |
| Recent meals (last 3) | Ephemeral (re-assembled each call) | ❌ Never |
| Situational data | Ephemeral (real-time signal) | ❌ Never |
| Conversation turns | Stored | ✅ Always |

**Decision rule:** if the data changes faster than the conversation does, make it ephemeral. Meal history today ≠ meal history last Tuesday. The user's name does not change — it belongs in semantic memory.

In [ ]:
def build_context_window_with_ephemeral(
    user_message: str,
    in_context_window: list[dict],
    meal_history: list[dict] | None = None,
    user_profile: dict | None = None,
    situational_data: dict | None = None,
    skill_text: str = '',
) -> list[dict]:
    """
    Section 5.6a: Makes the ephemeral/stored distinction explicit.

    Ephemeral layers (assembled fresh, never stored):
        user_profile, meal_history[-3], situational_data

    Stored layer (passed in from InContextMemory.get_window()):
        in_context_window

    After generation: add only the assistant reply to in_context_window,
    NOT the ephemeral layers.
    """
    import json
    messages: list[dict] = []

    # Ephemeral: stable user facts (re-fetched from SemanticMemory each call)
    if user_profile:
        messages.append({'role': 'user',
                          'content': f'[Context] User profile:\n{json.dumps(user_profile, indent=2)}'})
        messages.append({'role': 'assistant',
                          'content': 'Understood — I have your profile loaded.'})

    # Ephemeral: recent meals (last 3, re-assembled each call)
    if meal_history:
        recent = meal_history[-3:]
        history_note = 'Recent meals logged:\n' + '\n'.join(
            f"  - {m.get('food','?')} ({m.get('meal_type','?')})" for m in recent
        )
        messages.append({'role': 'user',      'content': history_note})
        messages.append({'role': 'assistant', 'content': "Got it — I'll factor in your recent meals."})

    # Ephemeral: real-time situational signal (valid this turn only)
    if situational_data:
        messages.append({'role': 'user',
                          'content': f'[Situational — this turn only]\n{json.dumps(situational_data, indent=2)}'})
        messages.append({'role': 'assistant',
                          'content': 'Noted — I will use this for my response.'})

    # Stored: actual conversation turns (persists turn-to-turn)
    messages.extend(in_context_window)

    # Current query: always last — highest attention weight
    messages.append({'role': 'user', 'content': user_message})
    return messages


# Demo — two turns showing ephemeral layers re-assembled each time
in_context: list[dict] = []  # stored — grows turn by turn
user_profile = {'name': 'Jordan', 'goal': 'Lose 5 kg', 'restrictions': 'Lactose intolerant'}
meal_history = [{'food': 'oats', 'meal_type': 'breakfast'}, {'food': 'salmon', 'meal_type': 'dinner'}]

for turn, user_msg in enumerate(['How am I doing on protein today?', 'What should I eat tomorrow?'], 1):
    messages = build_context_window_with_ephemeral(
        user_message=user_msg,
        in_context_window=in_context,
        meal_history=meal_history,
        user_profile=user_profile,
        situational_data={'current_time': '19:30', 'day_of_week': 'Tuesday'},
    )
    print(f'Turn {turn} — {len(messages)} messages in context')
    for i, msg in enumerate(messages):
        preview = str(msg['content'])[:55].replace('\n', ' ')
        stored = '(stored)' if msg in in_context else '(ephemeral)'
        if msg['content'] == user_msg:
            stored = '(current query)'
        print(f'  [{i}] {msg["role"]:10s} {stored:15s} | {preview}')

    # Only add the user + simulated reply to in_context — NOT ephemeral layers
    in_context.append({'role': 'user',      'content': user_msg})
    in_context.append({'role': 'assistant', 'content': f'[Simulated reply to turn {turn}]'})
    print(f'  → in_context now holds {len(in_context)//2} stored turn(s)\n')


### How context engineering connects to memory

The Mem0-based Diet Coach (Section 5.5) and `build_context_window` address different parts of the same problem:

- **Mem0 + `chat_turn`** decides *what to store and retrieve* — extracting facts from conversations, consolidating knowledge over time, and retrieving relevant memories at query time.
- **`build_context_window`** decides *how to present what was retrieved* — ordering layers by relevance, limiting recency to last-3 meals, using synthetic acknowledgement turns to prime the model.

In a production system you would combine both: use Mem0 for long-term memory management and `build_context_window` for deliberate context assembly before each LLM call. The memory stores provide the raw material; context engineering determines how that material is shaped into the model's view of the world.

## 5.7 · Chapter Summary

| Memory type | What we built | Production path |
|---|---|---|
| Semantic | JSON key-value store (5.2) | Redis / DynamoDB |
| Episodic | Session summaries + keyword retrieval (5.3) | Vector DB + embeddings |
| In-context | Sliding window with `max_turns` (5.4) | Same — already optimal |
| Long-term (Mem0) | LangChain + Mem0 memory client (5.5) | Mem0 managed service / self-hosted |
| Procedural | SKILL.md (Chapter 3) | Skill registry + semantic search |
| Context | `build_context_window` — layered assembly (5.6) | Same pattern; swap to streaming context with vector retrieval |

**What is next?** Chapter 6 — Communication: the protocols that let multiple agents talk to each other reliably.

---
*Mastering Agentic AI · Manning Publications*